# 🧹 Notebook 02 — Data Cleaning & Feature Engineering

**Smart Home Energy Consumption Prediction**

> Objective: Encode categorical features, engineer new time-based features, scale numerical data, and prepare the final feature matrix.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)
df = pd.read_csv('../data/Energy_consumption.csv', parse_dates=['Timestamp'])
print("Loaded:", df.shape)
df.head()

## 1. Initial Quality Check

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Duplicates ===")
print("Duplicate rows:", df.duplicated().sum())
print("\n=== Data Types ===")
print(df.dtypes)
print("\n=== Unique values — categorical columns ===")
for col in ['HVACUsage','LightingUsage','Holiday','DayOfWeek']:
    print(f"  {col}: {df[col].unique()}")

## 2. Feature Engineering — Datetime

In [ ]:
df['Hour']        = df['Timestamp'].dt.hour
df['Month']       = df['Timestamp'].dt.month
df['DayOfMonth']  = df['Timestamp'].dt.day
df['WeekOfYear']  = df['Timestamp'].dt.isocalendar().week.astype(int)
df['IsWeekend']   = df['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)
df['IsPeakHour']  = df['Hour'].between(17, 22).astype(int)   # Evening peak
df['Season']      = df['Month'].map({
    12:'Winter',1:'Winter',2:'Winter',
    3:'Spring',4:'Spring',5:'Spring',
    6:'Summer',7:'Summer',8:'Summer',
    9:'Autumn',10:'Autumn',11:'Autumn'
})

# Cyclical encoding for hour & month (preserves circular continuity)
df['Hour_sin']    = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos']    = np.cos(2 * np.pi * df['Hour'] / 24)
df['Month_sin']   = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos']   = np.cos(2 * np.pi * df['Month'] / 12)

print("New features added:", ['Hour','Month','DayOfMonth','WeekOfYear','IsWeekend',
      'IsPeakHour','Season','Hour_sin','Hour_cos','Month_sin','Month_cos'])
print("Shape after FE:", df.shape)

## 3. Encoding Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Binary encoding
df['HVAC_enc']     = (df['HVACUsage'] == 'On').astype(int)
df['Lighting_enc'] = (df['LightingUsage'] == 'On').astype(int)
df['Holiday_enc']  = (df['Holiday'] == 'Yes').astype(int)

# Ordinal day-of-week mapping (preserves order)
day_order = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
df['DayOfWeek_enc'] = df['DayOfWeek'].map(day_order)

# One-hot encode Season
df = pd.get_dummies(df, columns=['Season'], prefix='Season', drop_first=False)
# Convert boolean to int
season_cols = [c for c in df.columns if c.startswith('Season_')]
df[season_cols] = df[season_cols].astype(int)

print("Encoded features created.")
print("Season columns:", season_cols)
df[['HVAC_enc','Lighting_enc','Holiday_enc','DayOfWeek_enc'] + season_cols].head()

## 4. Feature Selection

In [ ]:
# Drop original raw columns no longer needed
drop_cols = ['HVACUsage','LightingUsage','Holiday','DayOfWeek','Timestamp']
df_model = df.drop(columns=drop_cols)

# Final feature list
target = 'EnergyConsumption'
feature_cols = [c for c in df_model.columns if c != target]

print(f"Total features: {len(feature_cols)}")
print("\nFeature list:")
for f in feature_cols:
    print(f"  • {f}")

## 5. Outlier Treatment (Capping at IQR)

In [ ]:
# Soft-cap with IQR (Winsorization) on numeric features only
numeric_features = ['Temperature','Humidity','SquareFootage','Occupancy','RenewableEnergy']

for col in numeric_features:
    q1 = df_model[col].quantile(0.01)
    q99 = df_model[col].quantile(0.99)
    before = df_model[col].describe()
    df_model[col] = df_model[col].clip(lower=q1, upper=q99)
    after = df_model[col].describe()
    print(f"{col}: clipped [{q1:.2f}, {q99:.2f}]")

print("\nOutlier capping complete.")

## 6. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df_model[feature_cols]
y = df_model[target]

# Train/test split BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# Scale only numerical columns
scale_cols = ['Temperature','Humidity','SquareFootage','Occupancy','RenewableEnergy',
              'Hour','Month','DayOfMonth','WeekOfYear']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

print("\nScaling complete (StandardScaler fitted on train only).")
print("X_train_scaled shape:", X_train_scaled.shape)

## 7. Save Processed Data

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)

# Save processed datasets
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save scaler
joblib.dump(scaler, '../models/scaler.pkl')

print("Saved:")
print("  ../data/X_train.csv")
print("  ../data/X_test.csv")
print("  ../data/y_train.csv")
print("  ../data/y_test.csv")
print("  ../models/scaler.pkl")
print(f"\nFinal feature matrix: {X_train_scaled.shape[1]} features, {len(X_train_scaled)+len(X_test_scaled)} records")

## 8. Preprocessing Summary

| Step | Action | Result |
|---|---|---|
| Missing values | None found | No action needed |
| Duplicates | None found | No action needed |
| Binary encoding | HVAC, Lighting, Holiday | 0/1 encoding |
| Day encoding | DayOfWeek | Ordinal 0–6 |
| Season encoding | One-hot | 4 columns |
| Datetime features | Hour, Month, etc. | +9 features |
| Cyclical encoding | Hour/Month | sin/cos pairs |
| Peak hour flag | 17:00–22:00 | Binary feature |
| Outlier capping | IQR 1%–99% | Soft winsorization |
| Scaling | StandardScaler | Train-only fit |

✅ **Processed data saved. Ready for model training.**